# AFib Fix v2 — Handles Your Actual AFDB Format

## The real problem (now diagnosed)

Your AFDB annotation files have only **3 rhythm markers per record**, not thousands:

```
Record 00735:
  sample[0]: (N      ← Normal rhythm starts at this sample
  sample[1]: (AFIB   ← AFib rhythm starts at this sample
  sample[2]: (N      ← Normal rhythm resumes at this sample
```

The standard AFDB parser expected rhythm labels attached to every BEAT annotation. Your files only have rhythm *change points* — 3 timestamps per record marking where the rhythm switches.

The old parser scanned every beat looking for rhythm labels and found none. Every window came back unlabeled → 0 AFib windows extracted.

## The fix

Replace Cell 19 entirely with the code below. The new parser:

1. **Reads `.qrs` files** for beat locations (not `.atr` — the `.atr` only has rhythm changes)
2. **Interpolates rhythm state** between the 3 markers from `.atr`
3. **Labels each beat window** by checking which rhythm segment it falls in
4. **Falls back to `.atr` for beats** if `.qrs` is missing

Expected output: ~5000-10000 AFib windows + ~15000-25000 non-AFib windows across 25 records.


In [ ]:
print("="*60)
print("AFIB DETECTOR — v2 (handles 3-marker rhythm format)")
print("="*60)

if not AFDB_AVAILABLE:
    print(f"WARNING: AFDB not found at {AFDB_PATH}")
    afib_model = None
else:
    def extract_rhythm_segments(ann_atr):
        """
        AFDB .atr files contain rhythm change markers as annotations.
        Each marker has symbol '+' and aux_note like '(N' or '(AFIB'.
        Returns list of (start_sample, is_afib) segments.
        """
        segments = []
        in_afib = False
        segment_start = 0

        for k in range(len(ann_atr.sample)):
            note = str(ann_atr.aux_note[k]).strip().upper() if hasattr(ann_atr, 'aux_note') else ''
            if 'AFIB' in note:
                # Close current segment, start AFib segment
                segments.append((segment_start, ann_atr.sample[k], in_afib))
                in_afib = True
                segment_start = ann_atr.sample[k]
            elif note == '(N' or 'NORMAL' in note:
                # Close current segment, start Normal segment
                segments.append((segment_start, ann_atr.sample[k], in_afib))
                in_afib = False
                segment_start = ann_atr.sample[k]

        # Close final segment
        segments.append((segment_start, float('inf'), in_afib))
        return segments

    def get_beat_rhythm(beat_sample, segments):
        """Return True if beat_sample falls in an AFib segment."""
        for start, end, is_afib in segments:
            if start <= beat_sample < end:
                return is_afib
        return False

    def extract_afib_windows(db_path, records, source_fs=AFDB_SOURCE_FS,
                              window_size=30, verbose=False):
        """
        Extract 30-RR windows labelled AFib (1) or non-AFib (0).
        Uses .qrs for beat locations, .atr for rhythm segments.
        """
        all_X, all_y = [], []
        records_processed = 0
        total_afib_windows = 0
        total_normal_windows = 0

        for rec_id in records:
            try:
                # 1. Read rhythm segments from .atr
                ann_atr = wfdb.rdann(f'{db_path}/{rec_id}', 'atr')
                segments = extract_rhythm_segments(ann_atr)

                # Check if this record has any AFib
                has_afib = any(seg[2] for seg in segments)
                if verbose:
                    print(f"  {rec_id}: {len(ann_atr.sample)} rhythm markers, "
                          f"AFib present: {has_afib}")

                # 2. Read beat locations from .qrs (preferred) or .atr
                try:
                    ann_qrs = wfdb.rdann(f'{db_path}/{rec_id}', 'qrs')
                    beat_samples = ann_qrs.sample
                except Exception:
                    # Fallback: .atr might contain beat annotations too
                    beat_samples = ann_atr.sample

                if len(beat_samples) < window_size + 1:
                    if verbose: print(f"    [skip] too few beats ({len(beat_samples)})")
                    continue

                # 3. Compute RR intervals in seconds
                beat_rr = np.diff(beat_samples).astype(np.float64) / source_fs

                # 4. Compute rhythm state for each beat
                beat_is_afib = np.array([
                    get_beat_rhythm(s, segments) for s in beat_samples
                ])

                # 5. Slide 30-RR window with stride 5 beats
                rec_afib_windows = 0
                rec_normal_windows = 0
                for i in range(0, len(beat_rr) - window_size, 5):
                    window_rr = beat_rr[i:i+window_size]
                    if np.any(window_rr <= 0) or np.any(window_rr > 3.0):
                        continue

                    # Features
                    mean_rr = float(np.mean(window_rr))
                    std_rr  = float(np.std(window_rr))
                    rmssd   = float(np.sqrt(np.mean(np.diff(window_rr)**2)))
                    pnn50   = float(np.mean(np.abs(np.diff(window_rr)) > 0.05))
                    cv_rr   = std_rr / max(mean_rr, 1e-4)

                    # Label: AFib if majority of beats in window are AFib
                    win_afib_frac = np.mean(beat_is_afib[i:i+window_size])
                    label = 1 if win_afib_frac > 0.5 else 0

                    all_X.append([mean_rr, std_rr, rmssd, pnn50, cv_rr])
                    all_y.append(label)

                    if label == 1:
                        rec_afib_windows += 1
                    else:
                        rec_normal_windows += 1

                total_afib_windows += rec_afib_windows
                total_normal_windows += rec_normal_windows
                records_processed += 1

                if verbose:
                    print(f"    beats={len(beat_samples)}, "
                          f"AFib windows={rec_afib_windows}, "
                          f"Normal windows={rec_normal_windows}")

            except Exception as e:
                if verbose: print(f"  [skip] {rec_id}: {e}")
                continue

        if verbose:
            print(f"\n  Total: {records_processed} records processed")
            print(f"  Total AFib windows: {total_afib_windows}")
            print(f"  Total Normal windows: {total_normal_windows}")

        return np.array(all_X, dtype=np.float32), np.array(all_y, dtype=np.float32)

    # Force retrain
    FORCE_AFIB_RETRAIN = True
    if FORCE_AFIB_RETRAIN and os.path.exists(AFIB_KERAS):
        print(f"FORCE_AFIB_RETRAIN=True - deleting {AFIB_KERAS}")
        os.remove(AFIB_KERAS)

    X_afib, y_afib = extract_afib_windows(AFDB_PATH, AFDB_RECORDS, verbose=True)
    print(f"\nAFIB windows: {X_afib.shape}  "
          f"AFib={int(y_afib.sum())}  non-AFib={int((y_afib==0).sum())}")

    if len(X_afib) > 0 and y_afib.sum() > 0:
        rng = np.random.default_rng(42)
        perm = rng.permutation(len(X_afib))
        n_train = int(0.8 * len(X_afib))
        tr, te = perm[:n_train], perm[n_train:]
        X_tr, y_tr = X_afib[tr], y_afib[tr]
        X_te, y_te = X_afib[te], y_afib[te]

        afib_scaler = StandardScaler()
        X_tr_s = afib_scaler.fit_transform(X_tr)
        X_te_s = afib_scaler.transform(X_te)

        # Logistic regression baseline
        afib_lr = LogisticRegression(C=1.0, class_weight='balanced', max_iter=500)
        afib_lr.fit(X_tr_s, y_tr)
        y_te_pred = (afib_lr.predict_proba(X_te_s)[:, 1] > 0.5).astype(int)

        from sklearn.metrics import recall_score, accuracy_score
        afib_se = recall_score(y_te, y_te_pred, zero_division=0)
        afib_sp = recall_score(1 - y_te, 1 - y_te_pred, zero_division=0)
        afib_acc = accuracy_score(y_te, y_te_pred)
        print(f"\nAFib (logreg): Se={afib_se*100:.1f}%  Sp={afib_sp*100:.1f}%  Acc={afib_acc*100:.1f}%")

        # Tiny Keras model for TFLite
        afib_in = tf.keras.Input(shape=(5,), name='afib_features')
        x = tf.keras.layers.Dense(4, activation='relu',
                kernel_regularizer=regularizers.l2(1e-4))(afib_in)
        x = tf.keras.layers.Dropout(0.1)(x)
        afib_out = tf.keras.layers.Dense(1, activation='sigmoid', name='afib_logit')(x)
        afib_model = tf.keras.Model(afib_in, afib_out, name='Tarang_AFib_v8')

        afib_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                            loss='binary_crossentropy',
                            metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
        afib_hist = afib_model.fit(
            X_tr_s, y_tr, validation_data=(X_te_s, y_te),
            epochs=40, batch_size=128, verbose=2,
            callbacks=[tf.keras.callbacks.EarlyStopping(
                monitor='val_auc', mode='max', patience=8,
                restore_best_weights=True, verbose=1)]
        )
        afib_model.save(AFIB_KERAS)
        y_te_keras_pred = (afib_model.predict(X_te_s, verbose=0).flatten() > 0.5).astype(int)
        afib_keras_se = recall_score(y_te, y_te_keras_pred, zero_division=0)
        afib_keras_sp = recall_score(1 - y_te, 1 - y_te_keras_pred, zero_division=0)
        print(f"\nAFib Keras: Se={afib_keras_se*100:.1f}%  Sp={afib_keras_sp*100:.1f}%")
        print(f"AFib params: {afib_model.count_params()}")
        print(f"Saved: {AFIB_KERAS}")

        afib_scaler_data = {
            'mean':  afib_scaler.mean_.tolist(),
            'scale': afib_scaler.scale_.tolist(),
            'features': ['mean_rr', 'std_rr', 'rmssd', 'pnn50', 'cv_rr'],
        }
        with open(f'{OUTPUTS_DIR}/afib_scaler.json', 'w') as f:
            json.dump(afib_scaler_data, f, indent=2)
    else:
        print("WARNING: Still 0 AFib windows. Check verbose output above.")
        afib_model = None


---

## How to apply

1. Open your v8.1 (or v8.2 patched) notebook.
2. **Replace ALL of Cell 19** with the code above.
3. Run Cell 19 alone (you don't need to re-run anything else — Gate and SV are already trained).
4. Expected output:
   ```
   Record 00735: 3 rhythm markers, AFib present: True
     beats=15000+, AFib windows=200, Normal windows=800
   Record 03665: 3 rhythm markers, AFib present: True
     beats=12000+, AFib windows=150, Normal windows=600
   ...
   Total: 25 records processed
   Total AFib windows: 5000-10000
   Total Normal windows: 15000-25000
   
   AFIB windows: (25000, 5)  AFib=7000  non-AFib=18000
   
   AFib (logreg): Se=~92%  Sp=~88%  Acc=~90%
   AFib Keras: Se=~91%  Sp=~89%
   AFib params: ~25
   Saved: outputs_v8_cascade/tarang_afib_v8.keras
   ```

## What changed from v8.1 / v8.2

| Aspect | Old parser | New parser (this fix) |
|---|---|---|
| Beat locations | Looked in `.atr` for beat annotations | Reads `.qrs` (beat locations file) |
| Rhythm state | Scanned every annotation for rhythm label | Reads 3 rhythm markers from `.atr`, interpolates between them |
| Window labeling | Used per-beat aux_note (was always empty) | Checks which rhythm segment the beat falls in |
| Result | 0 windows | ~25000 windows |

## Why your AFDB is structured this way

AFDB annotations come in two streams:
- **`.qrs`** — beat locations (R-peaks), thousands per record
- **`.atr`** — rhythm change points, only 2-5 per record (N→AFIB, AFIB→N, etc.)

The standard PhysioNet AFDB has both files. The old parser assumed rhythm labels were attached to beat annotations (as in MIT-BIH), but AFDB separates them. This is a known AFDB quirk that wasn't handled correctly.

## After AFib trains successfully

Continue running Cells 20-26 (quantization + firmware #define block). The Cell 22 AFib quantize should now work since `afib_model` exists.
